## Encapsulation (Protecting Internal State)

Encapsulation bundles an object's data and methods together while restricting direct external access to sensistive internal attributes.

Instead of letting outside code assign invalid values directly(such as negative bank balance or an unsafe storage temperature), we force changes to go through controlled methods or @property setters.


### Access Conventions in Python:

- Public(self.account_holder) : Readable and writable from anywhere

- Protected(self._temperature) : Single leading underscore signals "Inernal attribute - do not touch directly ourside this class of subclasses"

- Private(self.__pin_bash): Double leading underscore triggers name mangling (Python renames it internally to _ClassName__pin_hash), preventing accidental overrides or direct external reads

### Example 1 : Cold-chain Medical Storage Unit (@porperty + Private State)

Vaccine storage units must strictly stay between 2.0C and 8.0C unsing @property lets external code read and update unit.target_temp like a normal attribute while running validation behind the scenes.

In [3]:
class ColdStorageUnit:
    MIN_SAFE_TEMP = 2.0
    MAX_SAFE_TEMP = 8.0

    def __init__(self,unit_id:str, target_temp: float, override_code: str) -> None:
        self.unit_id = unit_id # publilc
        self.__override_code = override_code #private
        self.target_temp = target_temp # calls @target_temp.setter immediately

    # Getter: runs when reading unit.target_temp

    @property
    def target_temp(self) -> float:
        return self._target_temp
    
    # Setter: runs when assigning unit.target_temp = value
    @target_temp.setter
    def target_temp(self, value:float) -> None:
        if not (self.MIN_SAFE_TEMP <= value <= self.MAX_SAFE_TEMP):
            raise ValueError(
                f"Unsafe temperature {value}C: must stay between {self.MIN_SAFE_TEMP} and {self.MAX_SAFE_TEMP}"
            )
        self._target_temp = value
    

    # Controlled method to interact with private __override_code

    def verify_override(self, code:str) -> bool:
        return code == self.__override_code


fridge = ColdStorageUnit("MED-04", 4.5, "9912")
print(f"{fridge.unit_id} set to {fridge.target_temp}") # just correct output

fridge.target_temp = 6.0 # Passes validation (it will call the setter function)

# fridge.target_temp = 12.0  #raises valueerror
# print(fridge.__override_code) # Raise AttributeError

print("Override valid ?", fridge.verify_override("9912")) # output correct


MED-04 set to 4.5
Override valid ? True


## Some theory points for my understanding



### How Private Attributes Work Inside vs Outside the Class

- for a private attribute , we cannt read or write directly outside the class, but within inside the class it works

- inside the class, the name_mangled is retranslated always - so we can use it directly

- a controlled method acts as a gate keeper

In [4]:
class MetroCard:
    def __init__(self, card_id: str, pin: str) -> None:
        self.card_id = card_id # Public : anyone can read/change
        self.__pin = pin # Private : hidden from outside code

    #Controlled method: outside code cannot read self.__pin,
    #so it must ask this method to check a guess
    def verify_pin(self, entered_pin: str) -> bool:
        return entered_pin == self.__pin


    # Controlled method: outside code cannot writecard.__pin = "1234"

    #so it must go through our validation rule here

    def change_pin(self, old_pin: str, new_pin: str) -> str:

    
        if not self.verify_pin(old_pin):
            return "Failed: current PIN is incorrect"

        if len(new_pin) != 4 or not new_pin.isdigit():
            return "Failed: New PIN must be 4 digits"

        self.__pin = new_pin
        return "PIN updated successfully"


card = MetroCard("KML-1042", "4589")

# Outside access fails
# print(card.__pin) #AttributeError
#card.__pin = "0000" # Does not change the real private __pin

# Access through controlled methods works safely:

print(card.verify_pin("4589"))
print(card.change_pin("4589","99"))
print(card.change_pin("4589","7721"))
    
        

True
Failed: New PIN must be 4 digits
PIN updated successfully


## Traditional Getters and Setters (Plain Methods)

Suppose we also want to protect balance on the MetroCard so nobody can set card.balance = -500.

in languages like Java or C++ , we can make self._balance internal and write two normal methods:

- Getter(get_balance): Reads and returns the internal value
- Setter(set_balance): Validates the new value before saving it to the internal attribute.

In [ ]:
class MetroCardBasic:

    def __init__(self, card_id: str, balance: float) -> None:
        self.card_id = card_id
        self._balance = 0.0
        self.set_balance(balance) # Reuse validation during setup


    # Traditional Getter
    def get_balance(self) -> float:
        return self._balance

    # Traditional Setter
    def set_balance(self, amount: float) -> None:
        if amount < 0:
            raise ValueError("Card balance cannot be negative")
        if amount > 5000:
            raise ValueError("Maximum Metor card limit is 5000")

        self._balance = amount

c = MetroCardBasic("KMRL-1042", 200.0)
print(c.get_balance())

# to add  150, we have to call both methods with parenthesis

c.set_balance(c.get_balance() + 150.0) #This works, but calling this way is clunky compared to writing c.balance += 150
print(c.get_balance())

200.0
350.0


## Why @property exists (Pythonic Getters and Setters)

Python created the @porperty decorator to give us the best of both worlds:

- For the person using our class = It looks and feels like a normal attribute(card.balance and card.balance = 500 - no paranthesis needed)

- Under the hood: Python secretly runs our getter and setter methods every time someone reads or assigns card.balance




Here is how the two decorators connect together:

- @property above def balance(self); Marks this method as the **getter**. Whenever someone writes print(card.balance), Python calls this method.

- @balance.setter above def balance(self, amount): Marks this method as the **setter** for that exact property name(balance). Whenever someone writes card.balance = 500, Python passes 500 into amount and runs this method

In [9]:
class MetroCard:
    def __init__(self, card_id: str, balance: float) -> None:
        self.card_id = card_id
        self.balance = balance # Triggers @balance.setter automatically !


    # Getter : Runs whe we read card.balance

    @property
    def balance(self) -> float:
        return self._balance

    # Setter: Runs when we assign card.balance = value

    @balance.setter
    def balance(self, amount: float) -> None:
        if amount < 0:
            raise ValueError("card balance cannot be negative")
        if amount > 5000:
            raise ValueError("Maximum metro card limit is 5000")

        self._balance = amount

card = MetroCard("KMRL", 200.0)

# Looks like normal attribute access, but runs the @property getter:
print(card.balance)

# Looks like normal assignment, but runs @balance.setter validation:
card.balance += 150.0 #Reads 200.0 -> adds 150 -> validates 350 -> then saves to _balance
print(card.balance)

200.0
350.0


## IMP

this may feel very complicated - 

_balance is used as the internal variable that holds the actual data, while balance is the interfave through which outside code interactis with it.

We use self.balance inside the getter and setter to access the stored data, and self.balance when we want the property mechanism to handle access or assignment.


so the sambavam is there is no big difference between the balance and _balance , it's just like we use two variables A and B, just names

it hav eno control or relation on the property or getter or setter , so why we do this is to control assigning values to the attribute of the class

like if we don't use setter and getter - we can assign an invalid balance value (-500) to the balance, but if setup setter and getter  , we can assign the value in normal way as card.balance = -500 , but this -500 value will be gone to setter first

then the setter validates and it set the value to the balance attribute - if only if the value we sent is valid one (which checked by the setter)


### why validation as seperate function ?

so if we do that we can assign or read in normal way as card.balance

## Abstraction (Enforcing Contracts and Hiding complexity)

Abstraction hides messy implementation details and exposes only a clean, predictable interface - defining what an object must do without forcing callers to care how it does it 


In Python, we build formal abstractions using built-in abc (Abstract Base Classes) module

- Inherit from ABC

- Decorate required methods with @abstractmethod

- Key Benefit: python blocks anyone from instantiating an abstract class directly or instantiating a child class that forgot to implement any @abstractmethod catching missing logic immediately instead of failing halfway through execution




### Example 2 : Logisitics and Freight Rate Engine

A warehouse checkout system only needs two things from any courier partner.

calculate_shipping_cost() and estimated_days(). whether the cost comes from volumetric air weight or road distance is hidden inside each subclass

In [11]:
from abc import ABC, abstractmethod

class ShippingCarrier(ABC):

    def __init__(self, weight_kg: float, distance_km: float) -> None:
        self.weight_kg = weight_kg
        self.distance_km = distance_km

    @abstractmethod
    def calculate_shipping_cost(self) -> float:
        """Must return total shipping charge in INR"""
        pass

    @abstractmethod
    def estimated_days(self) -> int:
        """ Must return estimated delivery duration in days."""
        pass

    # Concrete helper method shared by all carriers
    def quote_summary(self) -> str:
        cost = self.calculate_shipping_cost()
        days = self.estimated_days()

        return f"{self.__class__.__name__}: INR {cost:.2f} (Est. {days} days)"


class SurfaceCargo(ShippingCarrier):
    def calculate_shipping_cost(self) -> float:
        return 50.0 + (self.weight_kg * 12.0) + (self.distance_km * 0.15)

    def estimated_days(self) -> int:
        return max(2, int(self.distance_km // 400)+1)

class AirExpress(ShippingCarrier):
    def calculate_shipping_cost(self) -> float:
        return 180.0 + (self.weight_kg * 45.0) + (self.distance_km * 0.40)

    def estimated_days(self) -> int:
        return 1 if self.distance_km <= 1500 else 2


#carrier  = ShippingCarrier(5.0, 800) #TypeError: can't instntiate abstract class !


options: list [ShippingCarrier] = [ 
    SurfaceCargo(weight_kg=4.5, distance_km=1200),
    AirExpress(weight_kg=4.5,distance_km=1200),
]

for option in options:
    print(option.quote_summary())

SurfaceCargo: INR 284.00 (Est. 4 days)
AirExpress: INR 862.50 (Est. 1 days)


Ok, so as you can see in this code it's an example of abstraction - in the parent class - we can clearly see that there is no code part or implementation part on calculatinig the shipping days and cost

the implementation part is done by the child classes that inherits from the parent class - parent class doesn't want to know how the shipping and estimated days has to be calculated - it just asks for it

the child class inheriting the parent class proviedes those processed infos -which have the hard implementation logic


- so this mainly for code reusability - by doing this - we can write that common part within parents class like quote_summary() function , while the child classes doing their own specific implementaion 

which is differnt for a shipcarrier and aeroplane carrier and we can even have a Dronecarrier later - without affecting or touching the parent class


### why @abstractmethod 


because if someone creates a carrier but forgets to impelement the cost calculation

without @abstractmethod, python would allow this class to be instantiated, assuming the parent methods existed as ordinary methods.

The object could then use incomplete or inappropriate implementations

but with @abstractmethod

Python enforces the requirement  - If a subclass fails to implement the abstract method, then callig the class or instantiating raise a TypeError instead of allowing an incomplete carrier to be created.



**ABC makes the class an abstract base class and @abstractmethod marks methods that subclasses must implement before they can be instantiated**

## How Encapsulation and Abstraction work together

- Abstraction : defines the public dashboard( the buttons exposed to the user.)

- Encapsulation : locks the engine hood so outside code cannot tamper with internal wiring and break the system


# Remember this way

- **Abstraction** : Define what something must be able to do, without specifying how.

- **Inheritance** : Reuse or extend behavior from another class

- **Polymorphism** : Use different objects through the same interface, letting each behave differently

- **Encapsulation** : Control how an object's data and internal behavior are accessed

## SMARTPHONE BATTERY


In [ ]:
class SmartphoneBattery:
    def __init__(self, serial_number: str, charge_percent: int) -> None:
        self.__serial_number = serial_number
        self.charge_percent = charge_percent


    @property
    def charge_percent(self):
        return self._charge_percent

    @charge_percent.setter
    def charge_percent(self, charge: int) -:
        if not 0 <= charge <= 100:
            raise ValueError("Charge should be between 0 and 100")

        self._charge_percent = charge
        

    def get_masked_serial(self) -> str:
        return "*"*(len(self.__serial_number)-4) + self.__serial_number[-4:]


ab = SmartphoneBattery("12345678",65)
print(ab.charge_percent)
print(ab.get_masked_serial())
ab.charge_percent = 123

65
****5678


ValueError: Charge should be between 0 and 100

# Abstraction

Abstraction means hiding complex internal implementation details and exposing only a clean, mandatory set of actions (a contract) to whoever uses the class.

-**Real-world analogy**: When we use a biometric attendance machine at an office, we only place our finger on the scanner. We do not care whether the sensor uses optical light, capacitive ridges, or ultrasonic waves underneath - the machine guarantees one standard interface: verify_identity()

In Python: Abstraction lets a parent class say: "Any subclass that inherits from me must implement these specific methods, otherwise Python will refuse to create an object from it"



## Why not just use a regular parent class ?

with normal inheritance, if a developer creates a new subclass and forgets to implement a required method, Python will not complain when the object is created- the program will only crash later in production when that missing method finally gets called.


python's builtin abc (Abstract Base Classes) module prevents this mistake at object creation time:

1) Inherit the parent class from ABC

2) Place @abstractmethod above any method that every child class is forced to implement

3) Python immediately blocks:
    - Creating an object directly from the abstract parent class.
    - Creating an object from any child class that missed even one @abstractmethod

### Example 1: Office Biometric Access Control

Every access terminal(FingerprintScanner, RFIDCardReader) validates an employee differently under the hood, while the office security gate expects every terminal to guarantee authenticate() and log_entry()

In [30]:
from abc import ABC, abstractmethod

class AccessTerminal(ABC):

    def __init__(self, gate_name: str) -> None:
        self.gate_name = gate_name

    #Abstract methods: blueprints with NO implementation here
    #Every child class MUST write its own version

    @abstractmethod
    def authenticate(self, credential:str) -> bool:
        pass

    @abstractmethod
    def hardware_status(self) -> str:
        pass

    # Concrete method: shared normal method inherited by all terminals

    def process_entry(self, employee_name: str, credential: str) -> str:
        if self.authenticate(credential):
            return f"[{self.gate_name}] Access GRANTED to {employee_name}"
        return f"[{self.gate_name}] Access DENIED TO {employee_name}"


class FingerprintScanner(AccessTerminal):
    def authenticate(self, credential:str) -> bool:
        #Simulates matching a biometric hash template
        return credential.startswith("fp_hash") and len(credential) == 16

    def hardware_status(self) -> str:
        return "Optical sensor online (99.4%)"

class RFIDCardReader(AccessTerminal):
    def __init__(self, gate_name: str, valid_badges: set[str]) -> None:
        super().__init__(gate_name)
        self.valid_badges = valid_badges

    def authenticate(self, credential:str) -> bool:
        return credential in self.valid_badges

    def hardware_status(self) -> str:
        return f"NFC coil active ({len(self.valid_badges)} badges registrered)"


# Concrete cubclasses work once all @abstractmethods are implemented:

main_door = FingerprintScanner("Server Room")
side_gate = RFIDCardReader("Main Entrance", {"BADGE-101", "BADGE-102"})

print(main_door.process_entry("Devanath", "fp_hash_12354345"))
print(side_gate.process_entry("Alex", "BADGE-999"))
print(side_gate.hardware_status())

# Trying to instantiate the abstract class directly raises TypeError:
gate = AccessTerminal("Main Lobby")

[Server Room] Access GRANTED to Devanath
[Main Entrance] Access DENIED TO Alex
NFC coil active (2 badges registrered)


TypeError: Can't instantiate abstract class AccessTerminal without an implementation for abstract methods 'authenticate', 'hardware_status'

In [3]:
from abc import ABC, abstractmethod


class DocumnetExporter(ABC):

    def __init__(self, filename: str) -> None:
        self.filename = filename


    @abstractmethod
    def export(self, records: list[str]) -> str:
        pass

    @abstractmethod
    def file_extension(self) -> str:
        pass


class CSVExporter(DocumnetExporter):

    def export(self, records:list[str]) -> str:
        return f"Saved {len(records)} rows to {self.filename}.csv"

    def file_extension(self):
        return "csv"

class PDFExporter(DocumnetExporter):

    def export(self, records):
        return f"Saved {len(records)} rows to {self.filename}.pdf"

    def file_extension(self):
        return "pdf"

try:
    doc = DocumnetExporter("super.img")
except TypeError as e:
    print(e)


doc = PDFExporter("cat.img")

print(doc.export(["ab.img","rps.img"]))
print(doc.file_extension())

cs = CSVExporter("supa.txt")
print(cs.export(["abc","sir"]))
print(cs.file_extension())


Can't instantiate abstract class DocumnetExporter without an implementation for abstract methods 'export', 'file_extension'
Saved 2 rows to cat.img.pdf
pdf
Saved 2 rows to supa.txt.csv
csv
